# Variante B: previsão rolling de um passo

Na variante B cada previsão é de um passo, e o valor real chega logo depois. Os parâmetros ficam congelados:

1. prever `t+1` usando só o que foi observado até `t`;
2. observar `y[t+1]`;
3. atualizar a janela de resíduos com o erro da base naquele instante;
4. avançar, **sem chamar `fit`**.

A seleção usa o erro de um passo na validação. A seleção dinâmica de corretores só é permitida nesta variante.

In [1]:
import autohybts

print("AutoHybTS", autohybts.__version__)

AutoHybTS 0.0.1


## 1. Experimento na variante B

In [2]:
from autohybts import AutoHybridExperiment
from autohybts.datasets import get_data

y = get_data("synthetic_daily")
exp = AutoHybridExperiment(fh=1, variant="B", test_size=28, session_id=42).fit(y)
result = exp.compare_models()
result.leaderboard

,name,model,family,strategy,base,corrector,residual_source,window,alpha,mean_rank,...,MAE,RMSE,sMAPE,coverage,n_units,n_failed,n_skipped,fit_seconds,predict_seconds,is_champion
candidate_id,,,,,,,,,,,,,,,,,,,,,
f197a4a852eefc42b77e9d063dff3c84cf252feda2deb62c878faf84713a4449,arima,arima,statistical,standalone,arima,NaN,NaN,NaN,NaN,1.0,...,0.641927,0.641927,0.009300,1.0,3,0,0,22.988849,0.016920,False
0ec158f3b6ec517f088da25b7c2f75c10bfe627d7bf5437f0622b8ff36cca85a,arima-linear,"arima + linear [w=7, alpha=0.5, in_sample]",residual,residual_additive,arima,linear,in_sample,7.0,0.5,2.0,...,0.826935,0.826935,0.012094,1.0,3,0,0,0.688075,0.028472,False
dcc94aa07bedae7630b1f38c2570937852793c439f560bfd9098cbedf9650690,arima-linear,"arima + linear [w=7, alpha=0.5, historical_oof...",residual,residual_additive,arima,linear,historical_oof_1step,7.0,0.5,3.0,...,0.877855,0.877855,0.012860,1.0,3,0,0,7.849259,0.016726,False
50c3b10b84c20c32b6b898008596592cb4f651f3f73110b54230461381267f3e,arima-linear,"arima + linear [w=7, alpha=1.0, historical_oof...",residual,residual_additive,arima,linear,historical_oof_1step,7.0,1.0,4.0,...,1.151502,1.151502,0.016973,1.0,3,0,0,0.915692,0.029096,False
f52c951976d5bd5418cf8267e574cbb23673d633f248ab5aecf2fa4ae06270e5,linear-linear,"linear(lags=7) + linear [w=7, alpha=1.0, histo...",residual,residual_additive,linear,linear,historical_oof_1step,7.0,1.0,5.0,...,1.554794,1.554794,0.023039,1.0,3,0,0,1.476695,0.006458,False
1984f459ddd73973f8dee0312933d0c5d4a4ae9ff5e3e8a400a30bd0b87baff5,linear-linear,"linear(lags=7) + linear [w=7, alpha=0.5, histo...",residual,residual_additive,linear,linear,historical_oof_1step,7.0,0.5,6.0,...,1.634006,1.634006,0.024156,1.0,3,0,0,0.593906,0.003110,False
c05fbfc4191206898692474cb6edb6c95948dcd38fd203e4d6be47b604035efe,linear,linear(lags=7),linear,standalone,linear,NaN,NaN,NaN,NaN,7.0,...,1.713217,1.713217,0.025269,1.0,3,0,0,4.592492,0.003271,True
3e1cf077f43789200b40b04ab5b6098eb7f916132b72ca9bbd6f6f8eba5f3d02,arima-linear,"arima + linear [w=7, alpha=0.5, historical_oof...",residual,residual_additive,arima,linear,historical_oof_multistep,7.0,0.5,8.0,...,1.791878,1.791878,0.027376,1.0,3,0,0,17.518330,0.019916,False
a02b2e3d287a824a49ab59fb6ba512b63b07cdd1f99e4dcfaef443617e0f77f2,knn,knn(lags=7),kernel,standalone,knn,NaN,NaN,NaN,NaN,9.0,...,1.832480,1.832480,0.027126,1.0,3,0,0,0.294479,0.008901,False


## 2. Finalizar e avaliar no teste selado

In [3]:
final = exp.finalize_model(result.best)
exp.evaluate_external(final).metrics

,MASE,MAE,RMSE,sMAPE
model,,,,
linear(lags=7),0.569762,0.893768,1.064695,0.012715


## 3. Streaming em produção

O mesmo modelo, recebendo observações novas uma a uma. Aqui simulamos a chegada dos dados com as últimas 28 observações da série.

In [4]:
import pandas as pd

stream = final.stream()
rows = []
for timestamp, actual in y.iloc[-28:].items():
    ticket = stream.predict_next()
    rows.append({"timestamp": timestamp, "y_pred": ticket.prediction, "y_true": actual})
    stream.observe(actual)  # só agora o valor real fica disponível

stream_log = pd.DataFrame(rows).set_index("timestamp")
stream_log["abs_error"] = (stream_log["y_true"] - stream_log["y_pred"]).abs()
stream_log

,y_pred,y_true,abs_error
timestamp,,,
2025-12-03,72.055404,72.429605,0.374201
2025-12-04,69.618670,71.246072,1.627402
2025-12-05,68.011756,70.065617,2.053861
2025-12-06,68.278524,67.422345,0.856179
2025-12-07,67.611119,67.361921,0.249198
2025-12-08,69.149156,70.737740,1.588583
2025-12-09,72.734916,73.033376,0.298460
2025-12-10,73.540131,74.455786,0.915654
2025-12-11,72.981826,72.770706,0.211121
